# BN â†’ EN Bulk Translation (Qwen2.5-3B-Instruct, base model)
Takes an input file â€” either a `test_sample_stratified*.parquet` (from the
sample-creation notebook) or an `.xlsx` with columns `ben`, `ref_en`,
`source` â€” normalizes it, runs every row through **Qwen2.5-3B-Instruct**
(zero-shot, no fine-tuning), and saves an output Excel file with columns:
`ben`, `ref_en`, `source`, `translated_en`.

**Before running:**
1. Kaggle â†’ Settings â†’ Accelerator â†’ **GPU T4 x2** (needed for the model).
2. Kaggle â†’ Add-ons â†’ Secrets: `HF_TOKEN`.
3. Internet: **On**.
4. Attach your input file via **Add Data â†’ Upload** (either the `.parquet`
   from the sample-creation notebook, or your own `.xlsx`), then set
   `INPUT_FILE` below to its path under `/kaggle/input/...`.


## 1. Install dependencies

In [ ]:
# Install extra packages. Use Kaggle's pre-installed torch.
# (torch==2.0.1+cu117 wheels were removed from PyTorch index — do not pin them.)
get_ipython().system("pip install -q pandas pyarrow openpyxl transformers accelerate bitsandbytes")


## 2. Secrets + auth

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as _e:
    HF_TOKEN = None
    print(f"Warning: Could not load HF_TOKEN ({_e})")
    print("Proceeding without authentication — works for public models like Qwen2.5-3B-Instruct.")

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Authenticated with HuggingFace.")
else:
    print("No HF auth — using public model access.")


## 3. Set your input file path

Point this at the file you attached under **Add Data**. Accepts either:
- a `.parquet` file (e.g. `test_sample_stratified__from__....parquet`)
- an `.xlsx` file with columns `ben`, `ref_en` (optional), `source` (optional)


In [ ]:
# --- Automation override (added for UI orchestration) ---
# If run_config.json is present (written by the backend orchestrator), read
# RUN_LABEL, MAX_ROWS, and locate the uploaded input file automatically.
# Falls back to manual values below if not present.
import json
import os as _os
import glob as _glob

_RUN_CONFIG_PATH = "/kaggle/input/*/run_config.json"
_matches = sorted(_glob.glob(_RUN_CONFIG_PATH))

_auto_run_label = None
_auto_input_file = None
_auto_max_rows = None

if _matches:
    _config_path = _matches[-1]
    with open(_config_path) as f:
        _cfg = json.load(f)
    _auto_run_label = _cfg.get("RUN_LABEL")
    _auto_max_rows = _cfg.get("MAX_ROWS")  # actual row count of the uploaded file
    _config_dir = _os.path.dirname(_config_path)
    _candidates = (_glob.glob(_config_dir + "/*.xlsx") +
                   _glob.glob(_config_dir + "/*.parquet"))
    if _candidates:
        _auto_input_file = _candidates[0]
    print(f"Automation override found: RUN_LABEL={_auto_run_label}, INPUT_FILE={_auto_input_file}, MAX_ROWS={_auto_max_rows}")
    print(f"  config dir: {_config_dir}")
    print(f"  all files: {_glob.glob(_config_dir + chr(47) + chr(42))}")
else:
    print("No run_config.json found - using manual values below (standalone mode).")


In [ ]:
import glob
import os
import pandas as pd

def _read_excel_robust(path):
    """Try openpyxl first (.xlsx), fall back to xlrd for old binary .xls."""
    try:
        return pd.read_excel(path, engine="openpyxl")
    except Exception:
        return pd.read_excel(path, engine="xlrd")

# EDIT THIS each run
RUN_LABEL = _auto_run_label or "base_qwen3b"

# EDIT THIS to your actual attached file path
INPUT_FILE = _auto_input_file or "/kaggle/input/your-dataset-name/test_sample_stratified__from__bn_en_pairs_parquet.parquet"

# When run via the UI, MAX_ROWS is set to the exact row count of the uploaded file.
# In standalone mode, bump this manually when you want a full run.
MAX_ROWS_SAFEGUARD = 10

# --- Check file exists FIRST, before any read attempt ---
if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Files currently under /kaggle/input:")
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if os.path.isfile(f):
            print(" ", f)
    raise FileNotFoundError(
        f"INPUT_FILE not found: {INPUT_FILE}. "
        "Update the path above and re-run."
    )

print(f"Found input file: {INPUT_FILE}")

# Safety check: refuse to process large files unless MAX_ROWS_SAFEGUARD is raised
_check_df = (_read_excel_robust(INPUT_FILE)
             if INPUT_FILE.endswith((".xlsx", ".xls"))
             else pd.read_parquet(INPUT_FILE))
print(f"Input file has {len(_check_df):,} rows. Safeguard limit: {MAX_ROWS_SAFEGUARD:,}")

if len(_check_df) > MAX_ROWS_SAFEGUARD:
    raise ValueError(
        f"SAFETY STOP: input file has {len(_check_df):,} rows, exceeding "
        f"MAX_ROWS_SAFEGUARD={MAX_ROWS_SAFEGUARD}. If this is intentional, "
        f"raise MAX_ROWS_SAFEGUARD and re-run."
    )


## 4. Load + normalize input to a common schema

Accepts `.parquet` or `.xlsx`. Required column: **`ben`** (Bengali text).
Optional columns: **`ref_en`** (reference English, if you have it),
**`source`** (provenance tag). Missing optional columns are filled with
`None`/`"unknown"` so downstream steps never break on their absence.

If the input is `.xlsx`, it is also **converted and saved as a matching
`.parquet`** for consistency with the rest of the pipeline.


In [ ]:

def load_and_normalize(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".parquet":
        df = pd.read_parquet(path)
    elif ext in (".xlsx", ".xls"):
        try:
            df = pd.read_excel(path, engine='openpyxl')
        except Exception:
            df = pd.read_excel(path, engine='xlrd')
    else:
        raise ValueError(f"Unsupported input file type: {ext}. Use .parquet or .xlsx")

    # Normalize column name casing/whitespace defensively
    df.columns = [str(c).strip().lower() for c in df.columns]

    if "ben" not in df.columns:
        # Be forgiving of common alternate names before giving up
        alt_names = {"bengali": "ben", "bn": "ben", "src_bn": "ben"}
        for alt, target in alt_names.items():
            if alt in df.columns:
                df = df.rename(columns={alt: target})
                break

    if "ben" not in df.columns:
        raise ValueError(
            f"Required column 'ben' not found. Columns present: {list(df.columns)}"
        )

    if "ref_en" not in df.columns:
        df["ref_en"] = None
    if "source" not in df.columns:
        df["source"] = "unknown"

    df = df[["ben", "ref_en", "source"]].copy()
    df["ben"] = df["ben"].astype(str).str.strip()
    df = df[df["ben"].str.len() > 0].reset_index(drop=True)

    return df, ext

input_df, input_ext = load_and_normalize(INPUT_FILE)
print(f"Loaded {len(input_df):,} rows from {INPUT_FILE}")
print(f"Schema: {list(input_df.columns)}")
input_df.head()


In [ ]:
# If the input was Excel, save a matching Parquet copy for consistency
if input_ext in (".xlsx", ".xls"):
    base_name = os.path.splitext(os.path.basename(INPUT_FILE))[0]
    CONVERTED_PARQUET_PATH = f"/kaggle/working/{base_name}__converted.parquet"
    input_df.to_parquet(CONVERTED_PARQUET_PATH, index=False)
    print(f"Converted Excel input to Parquet: {CONVERTED_PARQUET_PATH}")
else:
    print("Input was already Parquet â€” no conversion needed.")


## 5. Load Qwen2.5-3B-Instruct (base model, 4-bit)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Detect GPU CUDA compute capability.
# bitsandbytes 4-bit quantization requires sm_70+ (T4, A100, ...).
# P100 is sm_60 — current PyTorch (>=2.1) has NO CUDA kernels for sm_60.
# Run on CPU for P100 (slow but correct).
if torch.cuda.is_available():
    _cc = torch.cuda.get_device_capability(0)
    _gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {_gpu_name}  (CUDA capability {_cc[0]}.{_cc[1]})")
    _use_gpu = _cc[0] >= 7  # sm_70+ required for current PyTorch
else:
    _use_gpu = False
    print("No CUDA GPU detected.")

if _use_gpu:
    print("Using 4-bit (NF4) quantization on GPU.")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs = {"quantization_config": bnb_config, "device_map": "auto"}
else:
    # P100 sm_60 or no GPU: load on CPU in float32.
    # For 9-row test files this takes ~2-5 minutes.
    print("Running on CPU (P100 sm_60 not supported by installed PyTorch).")
    model_kwargs = {"dtype": torch.float32, "device_map": "cpu"}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    **model_kwargs,
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Model loaded.")


## 6. Translation prompt + batched bulk generation

Zero-shot chat-template prompting, greedy decoding (deterministic â€” useful
if you re-run this later on a fine-tuned model and want a fair comparison).


In [ ]:
def build_prompt(bengali_text: str) -> str:
    messages = [
        {"role": "system", "content": "You are a professional Bengali to English translator. Translate the given Bengali text into natural, fluent English. Output only the translation, nothing else."},
        {"role": "user", "content": bengali_text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def translate_batch(bengali_texts, max_new_tokens=256, batch_size=8):
    outputs = []
    for i in range(0, len(bengali_texts), batch_size):
        batch = bengali_texts[i:i + batch_size]
        prompts = [build_prompt(t) for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

        for j, out_ids in enumerate(gen):
            input_len = inputs["input_ids"][j].shape[0]
            new_tokens = out_ids[input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            outputs.append(text)

        if (i // batch_size) % 10 == 0:
            print(f"  translated {i + len(batch)}/{len(bengali_texts)}")

    return outputs

print("Ready to translate.")


## 7. Run bulk translation

In [ ]:
BATCH_SIZE = 8  # lower to 4 if you hit an out-of-memory error on your GPU

bengali_inputs = input_df["ben"].tolist()
print(f"Translating {len(bengali_inputs):,} rows with {MODEL_NAME}...")

translations = translate_batch(bengali_inputs, batch_size=BATCH_SIZE)

output_df = input_df.copy()
output_df["translated_en"] = translations

print("Done.")
output_df.head()


## 8. Save output Excel file

Schema: `ben`, `ref_en`, `source`, `translated_en` â€” this is the exact
input format the **scoring notebook** expects.


In [ ]:
base_name = os.path.splitext(os.path.basename(INPUT_FILE))[0]
OUTPUT_XLSX_PATH = f"/kaggle/working/{base_name}__translated__{RUN_LABEL}.xlsx"

output_df.to_excel(OUTPUT_XLSX_PATH, index=False)
print(f"Saved translated output to: {OUTPUT_XLSX_PATH}")
print(f"Rows: {len(output_df):,}  |  Columns: {list(output_df.columns)}")
print()
print("To download to your local machine:")
print("  1. Save this notebook version (Save Version, top right).")
print("  2. Open the notebook's 'Output' tab.")
print(f"  3. Find '{os.path.basename(OUTPUT_XLSX_PATH)}' and click the download icon.")
print()
print("Feed this file into the scoring notebook as INPUT_FILE.")


## Notes

- To translate with a **fine-tuned** model later instead of the base model,
  swap Section 5's `from_pretrained` call to load your base model + apply
  your LoRA/QLoRA adapter (`PeftModel.from_pretrained(model, adapter_path)`)
  before Section 6 â€” everything else in this notebook stays the same.
- `ref_en` passes through untouched if present in the input; it's optional
  and only used later by the scoring notebook.
- Lower `BATCH_SIZE` in Section 7 if you hit a CUDA out-of-memory error.
